In [1]:
! pip install gym

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 KB 28.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for gym: filename=gym-0.26.2-py3-none-any.whl size=827727 sha256=982073ab6ebf9d26f8483af7c1a3ca016dddd97468142515b5dc2c11d86a16f7
  Stored in directory: /home/hamano/.cache/pip/wheels/b9/22/6d/3e7b32d98451b4cd9d12417052affbeeeea012955d437da1da
Successfully built gym


In [3]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gym import spaces
from stable_baselines3 import PPO
from finrl.agents.stablebaselines3.models import DRLAgent

# ==========================================
# 1. MLFinLab特徴量を組み込んだカスタム環境の定義
# ==========================================
class MLFinLabFinRLEnv(gym.Env):
    """MLFinLabで前処理した特徴量をFinRLの状態空間に渡すためのカスタム環境"""
    metadata = {"render.modes": ["human"]}

    def __init__(self, df, features, initial_amount=1000000, trade_cost_pct=0.001):
        super(MLFinLabFinRLEnv, self).__init__()
        
        self.df = df.sort_values(['date', 'tic']).reset_index(drop=True)
        self.features = features  # MLFinLabで作った特徴量名のリスト
        self.initial_amount = initial_amount
        self.trade_cost_pct = trade_cost_pct
        
        # 日付の一覧と銘柄数
        self.dates = self.df['date'].unique()
        self.tics = self.df['tic'].unique()
        self.num_stocks = len(self.tics)
        
        # 状態空間（State Space）の次元数計算
        # [現金(1)] + [各銘柄の保有数量(N)] + [各銘柄の終値(N)] + [各銘柄のMLFinLab特徴量(N * 特徴量数)]
        self.state_dim = 1 + self.num_stocks + self.num_stocks + (self.num_stocks * len(self.features))
        
        # 投資行動空間（Action Space）: 各銘柄への売買指示（-1から1の範囲、後に保有上限等でスケール）
        self.action_space = spaces.Box(low=-1, high=1, shape=(self.num_stocks,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.state_dim,), dtype=np.float32)
        
        self.reset()

    def reset(self):
        self.current_step = 0
        self.cash = self.initial_amount
        self.holdings = np.zeros(self.num_stocks)
        self.portfolio_value = self.initial_amount
        self.asset_memory = [self.initial_amount]
        
        return self._get_state()

    def _get_state(self):
        # 現在のステップの日付データを抽出
        current_date = self.dates[self.current_step]
        current_df = self.df[self.df['date'] == current_date].sort_values('tic')
        
        # 終値とMLFinLab特徴量の取得
        closes = current_df['close'].values
        mlfinlab_feats = current_df[self.features].values.flatten() # 銘柄毎の特徴量を1次元に平坦化
        
        # 状態ベクトルの結合
        state = np.hstack([
            np.array([self.cash]),
            self.holdings,
            closes,
            mlfinlab_feats
        ]).astype(np.float32)
        
        return state

    def step(self, actions):
        # 1. 現在の資産価値を計算
        current_date = self.dates[self.current_step]
        current_df = self.df[self.df['date'] == current_date].sort_values('tic')
        closes = current_df['close'].values
        
        old_portfolio_value = self.cash + np.sum(self.holdings * closes)
        
        # 2. 行動（売買）の執行（簡易的な実装例。FinRLの標準的なロジックへの置き換えも可能）
        # actionsは -1(全売却) 〜 1(全力買い) の比率を想定
        for i in range(self.num_stocks):
            action = actions[i]
            target_holding = (action + 1) * 5 # 例: 最大10株保有とするためのスケール
            delta = target_holding - self.holdings[i]
            
            cost = delta * closes[i]
            execution_cost = abs(cost) * self.trade_cost_pct
            
            # 現金が足りるかチェック
            if self.cash - (cost + execution_cost) >= 0:
                self.holdings[i] = target_holding
                self.cash -= (cost + execution_cost)
        
        # 3. ステップを進める
        self.current_step += 1
        done = self.current_step >= len(self.dates) - 1
        
        # 4. 次のステップの資産価値と報酬計算
        if not done:
            next_date = self.dates[self.current_step]
            next_df = self.df[self.df['date'] == next_date].sort_values('tic')
            next_closes = next_df['close'].values
            
            new_portfolio_value = self.cash + np.sum(self.holdings * next_closes)
            reward = new_portfolio_value - old_portfolio_value # 資産の増減額を報酬とする
            
            self.portfolio_value = new_portfolio_value
            self.asset_memory.append(new_portfolio_value)
            next_state = self._get_state()
        else:
            reward = 0
            next_state = self._get_state()
            
        return next_state, reward, done, {}

# ==========================================
# 2. 実行・Agentへの投入パイプライン
# ==========================================
if __name__ == "__main__":
    # サンプル用：MLFinLabから出力されたと仮定するDataFrameの作成
    # ※ 実際にはここにMLFinLabの処理後データを入れます
    mock_data = {
        'date': pd.date_range(start='2026-01-01', periods=100).repeat(2),
        'tic': ['AAPL', 'MSFT'] * 100,
        'close': np.random.uniform(100, 200, 200),
        'frac_diff': np.random.normal(0, 1, 200),       # MLFinLab: 分数階微分
        'triple_barrier': np.random.choice([0, 1], 200) # MLFinLab: トリプルバリアラベルなど
    }
    df_mlfinlab = pd.DataFrame(mock_data)
    
    # MLFinLabから引き継ぎたい特徴量カラムの指定
    features_from_mlfinlab = ['frac_diff', 'triple_barrier']
    
    # カスタム環境のインスタンス化
    env_instance = MLFinLabFinRLEnv(df=df_mlfinlab, features=features_from_mlfinlab)
    
    # FinRLのラッパー等に適合させるため、ダミー環境辞書を作成（FinRLのDRLAgent用）
    env_config = {
        "price_array": None, # カスタム環境内部で処理するためNoneでOK
        "tech_array": None,
        "if_train": True
    }
    
    # 💡ここがFinRL（DRLAgent）への橋渡し
    # 独自のenvインスタンスを直接DRLAgentに認識させるためのラップ
    class FinRLWrappedEnv:
        def __init__(self, env): self.env = env
        def select_env(self): return self.env

    wrapped_env = FinRLWrappedEnv(env_instance)
    
    # FinRLのDRLAgentを使ってPPOを初期化・学習
    agent = DRLAgent(env = wrapped_env)
    ppo_model = agent.get_model("ppo")
    
    # 学習の実行
    trained_ppo = agent.train_model(model=ppo_model, tb_log_name="ppo_mlfinlab", total_timesteps=10000)
    print("Training Completed with MLFinLab features!")

{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 64}
Using cuda device


ValueError: The environment is of type <class '__main__.FinRLWrappedEnv'>, not a Gymnasium environment. In this case, we expect OpenAI Gym to be installed and the environment to be an OpenAI Gym environment.